# 03 - CNN baseline fuerte

En esta notebook se corre una primera CNN para la Parte 2 del TP.

La arquitectura es una versión reducida tipo AlexNet:

- bloques convolucionales;
- ReLU;
- MaxPooling;
- Batch Normalization opcional;
- Dropout en el clasificador;
- clasificador fully-connected final.

Se usa el split limpio ya generado en:

`data/splits/final_split_5fold.csv`

No se usa el test fijo para elegir hiperparámetros. La evaluación se hace mediante validación cruzada sobre `trainval`.

La corrida registra:

- TensorBoard en `runs/cnn/`;
- MLflow en `mlruns/`;
- métricas, curvas, matriz de confusión y reportes en `experiments/` y `results/`.

In [2]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()

if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("Repo root:", REPO_ROOT)

import json
from pathlib import Path

import pandas as pd

from cnn_training import run_cross_validation, save_experiment_outputs


Repo root: c:\Users\tomas\OneDrive\Documentos\MATERIAS ITBA\ELECTIVAS - CUATRIMESTRE X\MACHINE LEARNING y REDES NEURONALES EN BIOINGENIERÍA\skin-dataset-classification


In [3]:
cnn_baseline_config = {
    "experiment": "CNN_0_alexnetsmall_bn_dropout_wd_lightaug",
    "mlflow_experiment": "TP_skin_CNN",

    "model": "AlexNetSmall",
    "image_size": 128,
    "batch_size": 32,
    "epochs": 25,
    "lr": 3e-4,
    "optimizer": "adamw",

    "dropout": 0.3,
    "batch_norm": True,
    "weight_decay": 1e-4,
    "augmentation": "light",

    "early_stopping": True,
    "patience": 6,

    "tensorboard": True,
    "mlflow": True,
    "log_pytorch_model": True,
    "histogram_every": 5,

    "seed": 42,
}

cnn_baseline_config

{'experiment': 'CNN_0_alexnetsmall_bn_dropout_wd_lightaug',
 'mlflow_experiment': 'TP_skin_CNN',
 'model': 'AlexNetSmall',
 'image_size': 128,
 'batch_size': 32,
 'epochs': 25,
 'lr': 0.0003,
 'optimizer': 'adamw',
 'dropout': 0.3,
 'batch_norm': True,
 'weight_decay': 0.0001,
 'augmentation': 'light',
 'early_stopping': True,
 'patience': 6,
 'tensorboard': True,
 'mlflow': True,
 'log_pytorch_model': True,
 'histogram_every': 5,
 'seed': 42}

In [4]:
quick_config = cnn_baseline_config.copy()
quick_config["experiment"] = "CNN_QUICK_CHECK"
quick_config["epochs"] = 2
quick_config["early_stopping"] = False
quick_config["tensorboard"] = True
quick_config["mlflow"] = True
quick_config["log_pytorch_model"] = False

quick_output = run_cross_validation(
    config=quick_config,
    split_csv="data/splits/final_split_5fold.csv",
    folds_to_run=[0],
)

display(quick_output["fold_results"])
quick_output["summary"]

2026/06/17 22:34:07 INFO mlflow.tracking.fluent: Experiment with name 'TP_skin_CNN' does not exist. Creating a new experiment.


Device usado: cpu
Folds a correr: [0]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Parámetros entrenables: 2,736,009
[CNN_QUICK_CHECK] fold 0 | epoch 01/2 | train_acc=0.213 | val_acc=0.375 | val_f1=0.284
[CNN_QUICK_CHECK] fold 0 | epoch 02/2 | train_acc=0.333 | val_acc=0.479 | val_f1=0.399


,experiment,fold,best_epoch,val_loss,val_accuracy,val_macro_f1,val_balanced_accuracy,n_train,n_val
0,CNN_QUICK_CHECK,0,2,1.45471,0.479167,0.399031,0.456583,573,144


{'experiment': 'CNN_QUICK_CHECK',
 'model': 'AlexNetSmall',
 'image_size': 128,
 'batch_size': 32,
 'epochs': 2,
 'lr': 0.0003,
 'optimizer': 'adamw',
 'dropout': 0.3,
 'batch_norm': True,
 'weight_decay': 0.0001,
 'augmentation': 'light',
 'early_stopping': False,
 'folds_run': [0],
 'best_fold': 0,
 'num_trainable_params': 2736009,
 'val_accuracy_mean': 0.4791666666666667,
 'val_accuracy_std': 0.0,
 'val_macro_f1_mean': 0.39903130891502986,
 'val_macro_f1_std': 0.0,
 'val_balanced_accuracy_mean': 0.4565826330532212,
 'val_balanced_accuracy_std': 0.0,
 'mlflow_run_id': '2354725a5f6c4b068700c4a7a8fe5659'}

In [5]:
save_experiment_outputs(
    cv_output=quick_output,
    output_prefix="cnn_quick_check",
)

Archivos guardados:
- experiments\cnn_quick_check_fold_results.csv
- experiments\cnn_quick_check_history.csv
- experiments\cnn_quick_check_summary.json
- results\training_curves\cnn_quick_check_loss.png
- results\training_curves\cnn_quick_check_accuracy.png
- results\confusion_matrices\cnn_quick_check.png
- results\classification_reports\cnn_quick_check_classification_report.txt


In [6]:
cnn_baseline_output = run_cross_validation(
    config=cnn_baseline_config,
    split_csv="data/splits/final_split_5fold.csv",
    folds_to_run=None,  # None = correr los 5 folds
)

Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Parámetros entrenables: 2,736,009
[CNN_0_alexnetsmall_bn_dropout_wd_lightaug] fold 0 | epoch 01/25 | train_acc=0.213 | val_acc=0.375 | val_f1=0.284
[CNN_0_alexnetsmall_bn_dropout_wd_lightaug] fold 0 | epoch 02/25 | train_acc=0.333 | val_acc=0.479 | val_f1=0.399
[CNN_0_alexnetsmall_bn_dropout_wd_lightaug] fold 0 | epoch 03/25 | train_acc=0.396 | val_acc=0.375 | val_f1=0.326
[CNN_0_alexnetsmall_bn_dropout_wd_lightaug] fold 0 | epoch 04/25 | train_acc=0.478 | val_acc=0.507 | val_f1=0.480
[CNN_0_alexnetsmall_bn_dropout_wd_lightaug] fold 0 | epoch 05/25 | train_acc=0.492 | val_acc=0.569 | val_f1=0.533
[CNN_0_alexnetsmall_bn_dropout_wd_lightaug] fold 0 | epoch 06/25 | train_acc=0.518 | val_acc=0.583 | val_f1=0.530
[CNN_0_

2026/06/18 00:24:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [7]:
display(cnn_baseline_output["fold_results"])

,experiment,fold,best_epoch,val_loss,val_accuracy,val_macro_f1,val_balanced_accuracy,n_train,n_val
0,CNN_0_alexnetsmall_bn_dropout_wd_lightaug,0,7,1.155568,0.638889,0.592698,0.616713,573,144
1,CNN_0_alexnetsmall_bn_dropout_wd_lightaug,1,24,0.889846,0.673611,0.682417,0.679314,573,144
2,CNN_0_alexnetsmall_bn_dropout_wd_lightaug,2,6,1.125219,0.608392,0.607520,0.602334,574,143
3,CNN_0_alexnetsmall_bn_dropout_wd_lightaug,3,23,0.902170,0.657343,0.640948,0.656337,574,143
4,CNN_0_alexnetsmall_bn_dropout_wd_lightaug,4,19,1.019765,0.643357,0.637939,0.643265,574,143


In [8]:
cnn_baseline_output["summary"]

save_experiment_outputs(
    cv_output=cnn_baseline_output,
    output_prefix="cnn_0_alexnetsmall_bn_dropout_wd_lightaug",
)

Archivos guardados:
- experiments\cnn_0_alexnetsmall_bn_dropout_wd_lightaug_fold_results.csv
- experiments\cnn_0_alexnetsmall_bn_dropout_wd_lightaug_history.csv
- experiments\cnn_0_alexnetsmall_bn_dropout_wd_lightaug_summary.json
- results\training_curves\cnn_0_alexnetsmall_bn_dropout_wd_lightaug_loss.png
- results\training_curves\cnn_0_alexnetsmall_bn_dropout_wd_lightaug_accuracy.png
- results\confusion_matrices\cnn_0_alexnetsmall_bn_dropout_wd_lightaug.png
- results\classification_reports\cnn_0_alexnetsmall_bn_dropout_wd_lightaug_classification_report.txt


In [9]:
mlp_summaries = []

for path in Path("experiments").glob("mlp_*_summary.json"):
    with open(path, "r", encoding="utf-8") as f:
        summary = json.load(f)
    mlp_summaries.append(summary)

best_mlp = max(mlp_summaries, key=lambda x: x["val_accuracy_mean"])

comparison = pd.DataFrame([
    {
        "model": best_mlp["experiment"],
        "family": "MLP",
        "val_accuracy_mean": best_mlp["val_accuracy_mean"],
        "val_macro_f1_mean": best_mlp["val_macro_f1_mean"],
        "val_balanced_accuracy_mean": best_mlp["val_balanced_accuracy_mean"],
    },
    {
        "model": cnn_baseline_output["summary"]["experiment"],
        "family": "CNN",
        "val_accuracy_mean": cnn_baseline_output["summary"]["val_accuracy_mean"],
        "val_macro_f1_mean": cnn_baseline_output["summary"]["val_macro_f1_mean"],
        "val_balanced_accuracy_mean": cnn_baseline_output["summary"]["val_balanced_accuracy_mean"],
    },
])

display(comparison)

,model,family,val_accuracy_mean,val_macro_f1_mean,val_balanced_accuracy_mean
0,MLP_10_lower_lr_dropout03_bn_wd,MLP,0.613636,0.597504,0.614541
1,CNN_0_alexnetsmall_bn_dropout_wd_lightaug,CNN,0.644318,0.632304,0.639593


In [10]:
comparison_path = Path("experiments/mlp_vs_cnn_baseline_comparison.csv")
comparison.to_csv(comparison_path, index=False)
print("Guardado:", comparison_path)

Guardado: experiments\mlp_vs_cnn_baseline_comparison.csv
